# VD-HMM Training mit CmdStanPy (Parallel)

Dieses Notebook trainiert **nur VD-HMM** Modelle mit S=2,3,4.

**Vorteile von CmdStanPy:**

- ✓ Chains laufen parallel (statt sequenziell)
- ✓ ~2x schneller bei 2 Chains
- ✓ Bessere Performance

**Wichtig:** Lassen Sie parallel das HMM-Notebook laufen für maximale Effizienz!


In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path("..").resolve()))

import pickle
import numpy as np
from helpers import ModelData
import cmdstanpy

print(f"CmdStanPy Version: {cmdstanpy.__version__}")
print(f"CmdStan Path: {cmdstanpy.cmdstan_path()}")

CmdStanPy Version: 1.3.0
CmdStan Path: /Users/omidsedighi-mornani/.cmdstan/cmdstan-2.37.0


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load processed data
data_path = "../data/processed/processed_data.pkl"
model_data = ModelData.from_pickle(data_path)

print(model_data.summary())


ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 200
- Eval indices: 221

Benchmark data: Available
- Benchmark covariates shape: (921, 23)
- Columns: business_id, density, Checkin, category, chain...



In [3]:
# Configuration
stan_model_folder = Path("../data/stan_code")
fitted_model_folder = Path("../models")
fitted_model_folder.mkdir(parents=True, exist_ok=True)

print(f"✓ Configuration set")
print(f"  Stan models: {stan_model_folder}")
print(f"  Output folder: {fitted_model_folder}")

✓ Configuration set
  Stan models: ../data/stan_code
  Output folder: ../models


In [4]:
def prepare_stan_data(model_data: ModelData, S: int):
    """
    Bereitet die Daten für Stan vor.

    CmdStanPy braucht die Daten als JSON-kompatibles Dictionary.
    """
    stan_data = {
        "S": S,
        "N_total": int(model_data.n_total),
        "N_train": int(model_data.n_train),
        "N_obs": int(model_data.n_obs),
        "nCovs": int(model_data.n_covs),
        "Time": [int(x) for x in model_data.time],
        "Closed": [int(x) for x in model_data.closed],
        "Days": [float(x) for x in model_data.days],
        "Ratings": [int(x) for x in model_data.ratings],
        "Sentiment": [float(x) for x in model_data.sentiment],
        "Q": model_data.Q.tolist(),
        "R": model_data.R.tolist(),
        "X_test": model_data.X_test.tolist(),
    }
    return stan_data


# Test
test_data = prepare_stan_data(model_data, S=3)
print(f"✓ Stan data prepared")
print(f"  Keys: {list(test_data.keys())}")

✓ Stan data prepared
  Keys: ['S', 'N_total', 'N_train', 'N_obs', 'nCovs', 'Time', 'Closed', 'Days', 'Ratings', 'Sentiment', 'Q', 'R', 'X_test']


In [ ]:
def train_model_cmdstan(
    model_data,
    S,
    model_name="vdhmm",
    chains=2,
    parallel_chains=2,
    iter_warmup=500,
    iter_sampling=500,
    seed=42,
    adapt_delta=0.8,
    max_treedepth=10,
):
    """
    Trainiert ein Modell mit CmdStanPy.

    Parameters:
    -----------
    model_data : ModelData
        Daten für das Training
    S : int
        Anzahl der Hidden States (2-5)
    model_name : str
        'vdhmm' oder 'hmm'
    chains : int
        Anzahl der MCMC Chains
    parallel_chains : int
        Anzahl parallel laufender Chains (nutzt parallel_chains CPU Cores)
    iter_warmup : int
        Warmup Iterationen
    iter_sampling : int
        Sampling Iterationen (post-warmup)
    seed : int
        Random Seed
    adapt_delta : float
        Stan adapt_delta Parameter (0.8-0.99, höher = konservativer)
    max_treedepth : int
        Stan max_treedepth Parameter

    Returns:
    --------
    cmdstanpy.CmdStanMCMC : Fit-Objekt
    """
    assert model_name in ["vdhmm", "hmm"], f"Invalid model_name: {model_name}"
    assert S in range(2, 6), "S must be between 2 and 5"

    # Daten vorbereiten
    stan_data = prepare_stan_data(model_data, S)

    # Model file
    model_file = stan_model_folder / f"{model_name}.stan"
    if not model_file.exists():
        raise FileNotFoundError(f"Stan model not found: {model_file}")

    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} with S={S} states (CmdStanPy)")
    print(f"{'='*60}")
    print(f"Model file: {model_file}")
    print(f"\nConfiguration:")
    print(f"  Chains: {chains}")
    print(f"  Parallel chains: {parallel_chains}")
    print(f"  Warmup iterations: {iter_warmup}")
    print(f"  Sampling iterations: {iter_sampling}")
    print(f"  Total iterations: {iter_warmup + iter_sampling}")
    print(f"  Seed: {seed}")
    print(f"  Adapt delta: {adapt_delta}")
    print(f"  Max treedepth: {max_treedepth}")

    # Compile model
    print(f"\nCompiling model...")
    model = cmdstanpy.CmdStanModel(stan_file=str(model_file))
    print(f"✓ Model compiled")

    # Sample
    print(f"\nSampling...")
    fit = model.sample(
        data=stan_data,
        chains=chains,
        parallel_chains=parallel_chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
        show_progress=True,
    )

    print(f"\n✓ Sampling complete!")

    # Save model
    output_path = fitted_model_folder / f"{model_name}_{S}_cmdstan.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(
            {
                "fit": fit,
                "model_name": model_name,
                "S": S,
                "stan_data": stan_data,
                "summary": fit.summary(),
            },
            f,
        )

    print(f"✓ Model saved to {output_path}")

    # Diagnostics
    print(f"\n{'-'*60}")
    print("Diagnostics:")
    print(f"{'-'*60}")
    print(fit.diagnose())

    # Summary statistics
    print(f"\n{'-'*60}")
    print("Summary (first 20 parameters):")
    print(f"{'-'*60}")
    summary_df = fit.summary()
    print(summary_df.head(20))

    return fit


print("✓ Training function defined (CmdStanPy)")

✓ Training function defined (CmdStanPy)


## Training Configuration

**Paper-Standard:** 2 chains × 1000 iterations (500 warmup + 500 sampling)

**Für schnelles Testen:** 2 chains × 200 iterations (100 warmup + 100 sampling)


In [6]:
# Training Settings
SEED = 42
CHAINS = 4
PARALLEL_CHAINS = 4  # Nutzt 4 CPU Cores parallel
ITER_WARMUP = 500  # Paper: 500, Quick test: 100
ITER_SAMPLING = 500  # Paper: 500, Quick test: 100
ADAPT_DELTA = 0.8  # 0.8-0.95, höher falls divergent transitions
MAX_TREEDEPTH = 10  # 10-15

np.random.seed(SEED)

print("Training Configuration:")
print(f"  Seed: {SEED}")
print(f"  Chains: {CHAINS}")
print(f"  Parallel chains: {PARALLEL_CHAINS}")
print(f"  Warmup iterations: {ITER_WARMUP}")
print(f"  Sampling iterations: {ITER_SAMPLING}")
print(f"  Total iterations: {ITER_WARMUP + ITER_SAMPLING}")
print(f"  Total posterior samples: {CHAINS * ITER_SAMPLING}")

Training Configuration:
  Seed: 42
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 500
  Sampling iterations: 500
  Total iterations: 1000
  Total posterior samples: 2000


## Training VD-HMM Models (S=2, 3, 4)

Variable-Duration Hidden Markov Models mit zeitabhängigen Übergangswahrscheinlichkeiten.


In [ ]:
# Dictionary zum Speichern aller Modelle
trained_models = {}

# VD-HMM Training für S=2 bis S=4
for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# VD-HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model_cmdstan(
            model_data=model_data,
            S=S,
            model_name="vdhmm",
            chains=CHAINS,
            parallel_chains=PARALLEL_CHAINS,
            iter_warmup=ITER_WARMUP,
            iter_sampling=ITER_SAMPLING,
            seed=SEED,
            adapt_delta=ADAPT_DELTA,
            max_treedepth=MAX_TREEDEPTH,
        )

        trained_models[f"vdhmm_{S}"] = fit
        print(f"\n✓✓✓ VD-HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training VD-HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("VD-HMM Training Complete!")
print("=" * 60)

00:23:43 - cmdstanpy - INFO - compiling stan file /var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/tmpmkotd0e4/tmp_bmlfc9v.stan to exe file /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/stan_code/vdhmm




############################################################
# VD-HMM Training: S=2
############################################################


Training VDHMM with S=2 states (CmdStanPy)
Model file: ../data/stan_code/vdhmm.stan

Configuration:
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 500
  Sampling iterations: 500
  Total iterations: 1000
  Seed: 42
  Adapt delta: 0.8
  Max treedepth: 10

Compiling model...


00:24:08 - cmdstanpy - INFO - compiled model executable: /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/stan_code/vdhmm


✓ Model compiled

Sampling...


00:24:09 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]





chain 1:   0%|          | 1/1000 [00:00<16:23,  1.02it/s, (Warmup)]

chain 1:  20%|██        | 200/1000 [23:13<1:26:32,  6.49s/it, (Warmup)]




chain 1:  30%|███       | 300/1000 [28:16<57:15,  4.91s/it, (Warmup)]  


chain 1:  40%|████      | 400/1000 [32:30<39:44,  3.97s/it, (Warmup)]







chain 1:  50%|█████     | 501/1000 [38:00<30:54,  3.72s/it, (Sampling)]




chain 1:  60%|██████    | 600/1000 [43:34<23:48,  3.57s/it, (Sampling)]




chain 1:  70%|███████   | 700/1000 [49:33<17:54,  3.58s/it, (Sampling)]




chain 1:  80%|████████  | 800/1000 [55:39<12:01,  3.61s/it, (Sampling)]

chain 2: 100%|██████████| 1000/1000 [1:45:46<00:00,  6.35s/it, (Sampling completed)]

chain 3: 100%|██████████| 1000/1000 [1:45:46<00:00,  6.35s/it, (Sampling completed)]


chain 4: 100%|██████████| 1000/1000 [1:45:46<00:00,  6.35s/it, (Sampling completed)]

...

05:59:10 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 2 had 1 divergent transitions (0.2%)
	Chain 3 had 2 divergent transitions (0.4%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.



✓ Sampling complete!
✓ Model saved to ../models/vdhmm_3_cmdstan.pkl

------------------------------------------------------------
Diagnostics:
------------------------------------------------------------
Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
3 of 2000 (0.15%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete.


------------------------------------------------------------
Summary (first 20 parameters):
------------------------------------------------

05:59:20 - cmdstanpy - INFO - CmdStan start processing


                               Mean      MCSE    StdDev       MAD  \
lp__                  -45949.400000  0.183095  4.755170  4.581230   
pi[1]                      0.095444  0.000589  0.022683  0.022510   
pi[2]                      0.499061  0.000943  0.042473  0.042272   
pi[3]                      0.405495  0.000983  0.038510  0.039303   
tpm[1,1]                   0.921093  0.001982  0.088599  0.054104   
tpm[1,2]                   0.078907  0.001982  0.088599  0.054104   
tpm[2,1]                   0.592300  0.002253  0.082556  0.083579   
tpm[2,2]                   0.407700  0.002253  0.082556  0.083579   
tpm[3,1]                   0.018982  0.000383  0.020154  0.013424   
tpm[3,2]                   0.981018  0.000383  0.020154  0.013424   
intercept[1]               5.171150  0.027563  0.684509  0.654794   
intercept[2]               4.265380  0.021282  0.493242  0.483761   
intercept[3]               3.944980  0.020696  0.486502  0.485215   
lambda                     0.48194

chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]




chain 1:  30%|███       | 300/1000 [2:13:28<4:24:46, 22.69s/it, (Warmup)]



chain 1:  40%|████      | 400/1000 [2:34:39<3:07:31, 18.75s/it, (Warmup)]





chain 1:  50%|█████     | 501/1000 [3:03:44<2:31:22, 18.20s/it, (Sampling)]



chain 1:  60%|██████    | 600/1000 [3:23:35<1:44:02, 15.61s/it, (Sampling)]



chain 1:  70%|███████   | 700/1000 [3:45:46<1:13:42, 14.74s/it, (Sampling)]







chain 1:  80%|████████  | 800/1000 [4:05:05<45:27, 13.64s/it, (Sampling)]  

## Training Summary


In [ ]:
# Summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"\nSaved in: {fitted_model_folder}")

# List saved models
saved_models = sorted(fitted_model_folder.glob("vdhmm_*_cmdstan.pkl"))
print(f"\nSaved VD-HMM model files ({len(saved_models)}):")
for model_file in saved_models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name} ({size_mb:.2f} MB)")